[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/romiaprilian7406/sp500-ratio-cluster/blob/gColab/notebooks/sp500_ratclus_dataset_clean2.ipynb)

# Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import time
import warnings
import os

# Global Configurations

In [ ]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Ticker list Function

In [ ]:
# Mengambil daftar ticker S&P 500 terbaru dari dataset publik
def get_sp500_tickers():
    try:
        url = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/refs/heads/main/data/constituents.csv'
        df = pd.read_csv(url)
        # Mengubah format ticker (misal: BRK.B -> BRK-B) agar sesuai dengan yfinance
        tickers = [t.replace('.', '-') for t in df['Symbol'].tolist()]
        print(f"Berhasil mendapatkan {len(tickers)} ticker S&P 500")
        return tickers
    except Exception as e:
        print(f"Gagal mengambil ticker: {e}")
        return []

# Data Fetching Function

In [ ]:
def fetch_raw_data(ticker):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        if not info or len(info) < 5: return None

        data = {
            'ticker': ticker,
            'sector': info.get('sector', 'Unknown'),
            'marketCap': info.get('marketCap'),

            # Valuation
            'trailingPE': info.get('trailingPE'),
            'forwardPE': info.get('forwardPE'),
            'priceToBook': info.get('priceToBook'),
            'enterpriseToRevenue': info.get('enterpriseToRevenue'),

            # Profitability
            'returnOnEquity': info.get('returnOnEquity'),
            'returnOnAssets': info.get('returnOnAssets'),
            'profitMargins': info.get('profitMargins'),
            'operatingMargins': info.get('operatingMargins'),

            # Solvency
            'debtToEquity': info.get('debtToEquity'),
            'currentRatio': info.get('currentRatio'),
            'quickRatio': info.get('quickRatio'),

            # Volatility & Growth
            'beta': info.get('beta'),
            'revenueGrowth': info.get('revenueGrowth'),
            'earningsGrowth': info.get('earningsGrowth'),

            # Placeholder (akan dihitung manual nanti)
            'pegRatio': info.get('pegRatio')
        }
        return data

    except:
        return None

# Simple Data Cleaning & Preprocessing

In [ ]:
# Membersihkan dataset mentah dengan logika strict namun sesuai konteks Clustering, dengan penyesuaian kolom dan logika negatif
def clean_raw_dataset(df):
    df = df.copy()

    # 1. Drop Duplicates
    if 'ticker' in df.columns:
        df = df.drop_duplicates(subset=['ticker'], keep='first')

    # 2. Type Casting (Memastikan semua angka terbaca sebagai Float)
    # Sesuaikan list ini dengan kolom yang ada di dataset clustering Anda
    num_cols = [
        'marketCap', 'trailingPE', 'forwardPE', 'pegRatio', 'priceToBook',
        'enterpriseToRevenue', 'profitMargins', 'returnOnEquity', 'returnOnAssets',
        'operatingMargins', 'debtToEquity', 'currentRatio', 'quickRatio',
        'beta', 'revenueGrowth', 'earningsGrowth'
    ]

    # Filter hanya kolom yang benar-benar ada di DF
    cols_to_clean = [c for c in num_cols if c in df.columns]

    for col in cols_to_clean:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. Handle Infinite Values (Penting untuk K-Means)
    # Mengubah 'inf' (hasil pembagian nol) menjadi NaN agar bisa di-impute
    df = df.replace([np.inf, -np.inf], np.nan)

    # 4. Standardize Sector
    if 'sector' in df.columns:
        df['sector'] = df['sector'].fillna('Unknown').astype(str).str.strip().str.title()

    # 5. Strict Sanity Filter
    # Untuk Clustering, 'marketCap' adalah data wajib.
    # Rasio lain boleh NaN (karena akan  impute), tapi tanpa Market Cap, data tidak valid.
    subset_wajib = ['marketCap']
    # Pastikan kolom subset ada sebelum dropna
    valid_subset = [c for c in subset_wajib if c in df.columns]
    if valid_subset:
        df = df.dropna(subset=valid_subset)

    # 6. Logic Filter (Hanya Market Cap yang Wajib Positif)
    # TIDAK memfilter earningsGrowth/Margin negatif,
    # karena itu adalah fitur valid untuk cluster saham "Loss Making".
    if 'marketCap' in df.columns:
        df = df[df['marketCap'] > 0]

    # 7. Reorder Columns (Identity -> Fundamental)
    # Memastikan ticker dan sector ada di depan
    first_cols = ['ticker', 'sector', 'marketCap']
    other_cols = [c for c in df.columns if c not in first_cols]

    # Pastikan kolom yang diminta ada
    final_cols = [c for c in first_cols if c in df.columns] + other_cols
    df = df[final_cols]

    return df

# Main Execution

In [ ]:
tickers = get_sp500_tickers()
data_list = []

for ticker in tqdm(tickers):
    stock_data = fetch_raw_data(ticker)
    if stock_data:
        data_list.append(stock_data)
    time.sleep(0.5)

# Create Initial DataFrame
df_raw = pd.DataFrame(data_list)

Berhasil mendapatkan 503 ticker S&P 500


  0%|          | 0/503 [00:00<?, ?it/s]

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


In [ ]:
# REORDER COLUMNS
cols = ['ticker', 'sector', 'marketCap',
        'trailingPE', 'forwardPE', 'pegRatio', 'priceToBook', 'enterpriseToRevenue',
        'profitMargins', 'returnOnEquity', 'returnOnAssets', 'operatingMargins',
        'debtToEquity', 'currentRatio', 'quickRatio',
        'beta', 'revenueGrowth', 'earningsGrowth']

# Filter hanya kolom yang eksis
cols = [c for c in cols if c in df_raw.columns]
df_final = df_raw[cols]

# Preview
print("Top 5 Data Preview:")
display(df_final.head())

Top 5 Data Preview:


,ticker,sector,marketCap,trailingPE,forwardPE,pegRatio,priceToBook,enterpriseToRevenue,profitMargins,returnOnEquity,returnOnAssets,operatingMargins,debtToEquity,currentRatio,quickRatio,beta,revenueGrowth,earningsGrowth
0,MMM,Industrials,91377950720,27.405750,21.716455,None,19.692379,3.995,0.13700,0.72921,0.07971,0.24367,281.904,1.842,1.027,1.152,0.035,-0.375
1,AOS,Industrials,9242947584,17.778975,16.206387,None,4.979240,2.411,0.13851,0.28209,0.13878,0.18631,12.063,1.544,0.894,1.336,0.044,0.146
2,ABT,Healthcare,219854782464,15.869347,24.480621,None,4.310822,5.137,0.31880,0.30620,0.06793,0.19395,25.310,1.703,1.088,0.719,0.069,0.000
3,ABBV,Healthcare,396548079616,169.977260,18.497114,None,-150.080260,7.709,0.04004,1.37961,0.09585,0.35497,NaN,0.725,0.468,0.352,0.091,-0.887
4,ACN,Technology,162576990208,21.500822,18.551527,None,5.203229,2.291,0.11021,0.25509,0.11183,0.15220,25.380,1.420,1.301,1.262,0.073,-0.155


In [ ]:
# Hitung Manual PEG Ratio
if 'forwardPE' in df_raw.columns and 'earningsGrowth' in df_raw.columns:
    # Konversi ke numerik
    df_raw['forwardPE'] = pd.to_numeric(df_raw['forwardPE'], errors='coerce')
    df_raw['earningsGrowth'] = pd.to_numeric(df_raw['earningsGrowth'], errors='coerce')

    # Rumus: PEG = Forward PE / (Earnings Growth * 100)
    growth_percent = df_raw['earningsGrowth'] * 100
    growth_percent = growth_percent.replace(0, np.nan) # Hindari error pembagian nol

    # Isi kolom pegRatio
    df_raw['pegRatio'] = df_raw['forwardPE'] / growth_percent

    print(f"PEG Ratio dihitung manual, Terisi: {df_raw['pegRatio'].count()} baris")
else:
    print("Kolom forwardPE/earningsGrowth tidak ditemukan.")

PEG Ratio dihitung manual, Terisi: 440 baris


In [ ]:
if not df_raw.empty:
    # Masukkan df_raw ke fungsi cleaning
    # Hasilnya simpan ke variabel 'df' agar konsisten dengan cell analisis di bawahnya
    df = clean_raw_dataset(df_raw)

    print(f"Total Data Mentah : {len(df_raw)}")
    print(f"Total Data Bersih : {len(df)}")
else:
    df = pd.DataFrame()
    print("DataFrame kosong.")

Total Data Mentah : 502
Total Data Bersih : 502


# Simple EDA

In [ ]:
# SHAPE DATASET
rows, cols = df.shape
print(f"Dimensi Dataset")
print(f"Total Baris    : {rows}")
print(f"Total Kolom    : {cols}\n")

# INFORMASI
print(df.info(),"\n")

# Target variable di sini adalah 'label'
features_list = [col for col in df.columns if col != 'label']
print(f"Jumlah Fitur : {len(features_list)}")
print(f"Daftar Fitur : {features_list}\n")

# STATISTIK DESKRIPTIF FITUR NUMERIK
print(f"\nStatistik Deskriptif (Fitur Input)")
# Filter hanya numerik
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

try:
    # Tampilkan transpose agar mudah dibaca
    display(df[numeric_cols].describe().T.round(2))
except:
    print(df[numeric_cols].describe().T.round(2))

# MISSING VALUES REPORT
print(f"\nLaporan Missing Values:")
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Jumlah Hilang': missing_count,
    'Persentase (%)': missing_pct.map('{:.2f}%'.format)
})

# Tampilkan hanya kolom yang ada missing value
missing_only = missing_df[missing_df['Jumlah Hilang'] > 0]
if not missing_only.empty:
    print(missing_only)
else:
    print("Tidak ada missing values (Data Bersih!)")

Dimensi Dataset
Total Baris    : 502
Total Kolom    : 18

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ticker               502 non-null    object 
 1   sector               502 non-null    object 
 2   marketCap            502 non-null    int64  
 3   trailingPE           476 non-null    float64
 4   forwardPE            502 non-null    float64
 5   priceToBook          502 non-null    float64
 6   enterpriseToRevenue  498 non-null    float64
 7   returnOnEquity       473 non-null    float64
 8   returnOnAssets       502 non-null    float64
 9   profitMargins        502 non-null    float64
 10  operatingMargins     502 non-null    float64
 11  debtToEquity         448 non-null    float64
 12  currentRatio         484 non-null    float64
 13  quickRatio           484 non-null    float64
 14  beta                 500 non-nul

,count,mean,std,min,25%,50%,75%,max
marketCap,502.0,1.301584e+11,4.365484e+11,3.810635e+09,1.993472e+10,3.780192e+10,8.438777e+10,4.410386e+12
trailingPE,476.0,3.794000e+01,7.507000e+01,4.490000e+00,1.742000e+01,2.442000e+01,3.413000e+01,1.136670e+03
forwardPE,502.0,2.514000e+01,3.728000e+01,-1.886900e+02,1.358000e+01,1.964000e+01,2.672000e+01,5.005000e+02
priceToBook,502.0,1.410000e+00,5.238000e+01,-9.046400e+02,1.680000e+00,3.040000e+00,6.760000e+00,1.892500e+02
enterpriseToRevenue,498.0,5.150000e+00,6.090000e+00,-3.560000e+00,2.040000e+00,3.850000e+00,6.390000e+00,1.028500e+02
returnOnEquity,473.0,2.500000e-01,4.900000e-01,-2.070000e+00,9.000000e-02,1.500000e-01,2.900000e-01,5.640000e+00
returnOnAssets,502.0,7.000000e-02,6.000000e-02,-1.500000e-01,3.000000e-02,5.000000e-02,9.000000e-02,5.400000e-01
profitMargins,502.0,1.400000e-01,1.500000e-01,-1.400000e+00,7.000000e-02,1.300000e-01,2.100000e-01,7.100000e-01
operatingMargins,502.0,2.200000e-01,1.500000e-01,-8.900000e-01,1.200000e-01,2.000000e-01,3.100000e-01,9.800000e-01
debtToEquity,448.0,1.399300e+02,2.810300e+02,5.300000e-01,3.983000e+01,7.546000e+01,1.413500e+02,4.217210e+03



Laporan Missing Values:
                     Jumlah Hilang Persentase (%)
trailingPE                      26          5.18%
enterpriseToRevenue              4          0.80%
returnOnEquity                  29          5.78%
debtToEquity                    54         10.76%
currentRatio                    18          3.59%
quickRatio                      18          3.59%
beta                             2          0.40%
revenueGrowth                    2          0.40%
earningsGrowth                  57         11.35%
pegRatio                        62         12.35%


# Export Dataset

In [ ]:
file_name = 'sp500_ratclus_dataset.csv'
df.to_csv(file_name, index=False)
print(f"File tersimpan: {file_name}")

File tersimpan: sp500_ratclus_dataset.csv
